**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-dp-444115535128-sa-east-1-an/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed/'

print('Base Path: ', base_path)
print('Landing Path: ', landing_path)
print('Processed Path: ', processed_path)

# Define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

## Bronze

In [0]:
df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(f"{landing_path}/*.csv")
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

print("Total Rows: ", df.count())

display(df.limit(100))

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

### Moving files from source to processed directory

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

## Silver

In [0]:
df_orders = spark.read.table(f"{bronze_table}")

display(df_orders.limit(100))

**Transformations**

In [0]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())

# 2. Clean customer_id - keep numeric, else set 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
    .otherwise("999999")
    .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" - "July 01,2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date(F.col("order_placement_date"), "yyyy/MM/dd"),
        F.try_to_date(F.col("order_placement_date"), "dd-MM-yyyy"),
        F.try_to_date(F.col("order_placement_date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("order_placement_date"), "MMMM dd, yyyy")
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 6. Convert product id to string
df_orders = df_orders.withColumn("product_id", F.col("product_id").cast("string"))

In [0]:
# Check what's the maximum and minimum date
df_orders.agg(
    F.max("order_placement_date").alias("max_date"),
    F.min("order_placement_date").alias("min_date")
).show()

In [0]:
display(df_orders.limit(100))

In [0]:
df_products = spark.table("fmcg.silver.products")

display(df_products.limit(100))

In [0]:
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

display(df_joined.limit(100))

In [0]:
if not(spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option("delta.enableChangeDataFeed", "true").option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Gold

In [0]:
df_orders = spark.read.table(f"{silver_table}")

df_gold = df_orders.select(
    "order_id",
    df_orders["order_placement_date"].alias("date"),
    df_orders["customer_id"].alias("customer_code"),
    "product_code",
    "product_id",
    df_orders["order_qty"].alias("sold_quantity")
)

display(df_gold.limit(100))

In [0]:
gold_table

In [0]:
if not(spark.catalog.tableExists(gold_table)):
    df_gold.write.format("delta").option("delta.enableChangeDataFeed", "true").option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Merging Data source with parent

In [0]:
df_child = spark.read.table(f"{gold_table}").select("date", "product_code", "customer_code", "sold_quantity")

display(df_child.limit(100))

In [0]:
df_child.count()

In [0]:
# 1. Change the date to first day of the month
df_monthly = (
    df_child
    # 1. Get month start date
    .withColumn("month_start", F.trunc("date", "MM"))

    # 2. Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity")
    )
    
    # 3. Rename month_start back to 'date' to match the parent schema
    .withColumnRenamed("month_start", "date")
)

display(df_monthly.limit(100))

In [0]:
df_monthly.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code AND parent_gold.sold_quantity = child_gold.sold_quantity").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()